## Spatial preprocessing

### Steps

1. Import Spaceranger outputs  
2. Collect gene and spot-wise QC metrics and write to file 
3. Filter out spots by coordinate   
4. Remove the bottom 1% (by total counts) of spots per sample
5. Remove spots with less than 100 genes
6. Remove spots with >=30% mitochondrial genes per sample
7. Write preprocessed samples individually into `.h5ad` 
8. Concatenate all samples into one
9. Write the concatenated version into `h5ad`

In [4]:
import squidpy as sq
import scanpy as sc
import pandas as pd
import numpy as np
import os
pd.set_option('display.max_columns',400)
pd.set_option('display.max_categories',40)
import warnings
warnings.filterwarnings("ignore")


In [5]:
# Step 1: Read the .txt file as a pandas DataFrame
metadata_df = pd.read_excel("./metadata_batch_effect.xlsx", parse_dates=False)

# Step 2: Initialize an empty list to store AnnData objects
# Another list to store filter stats
filter_df = []

In [6]:
metadata_df.dtypes

library_id                   object
alignment                    object
desired_rc                    int64
actual_rc                     int64
dir                          object
xmin                          int64
xmax                          int64
ymin                          int64
ymax                          int64
hpx                           int64
vpx                           int64
slide                        object
area                         object
date                 datetime64[ns]
cytassist_ver                object
seq_batch                     int64
probe_date           datetime64[ns]
probe_lot                     int64
probe_group                   int64
cond                         object
cond_rep                     object
rep                           int64
age                           int64
sex                          object
bmi                           int64
packyears                   float64
FVC                         float64
DLCO                        

In [7]:
metadata_df['date']= metadata_df['date'].astype('str')

In [19]:
metadata_df['probe_date']= metadata_df['probe_date'].astype('str')

In [20]:
metadata_df.dtypes

library_id            object
alignment             object
desired_rc             int64
actual_rc              int64
dir                   object
xmin                   int64
xmax                   int64
ymin                   int64
ymax                   int64
hpx                    int64
vpx                    int64
slide                 object
area                  object
date                  object
cytassist_ver         object
seq_batch              int64
probe_date            object
probe_lot              int64
probe_group            int64
cond                  object
cond_rep              object
rep                    int64
age                    int64
sex                   object
bmi                    int64
packyears            float64
FVC                  float64
DLCO                 float64
6MWD                 float64
outcome_long          object
tte_long               int64
outcome_24m           object
tte_24m                int64
lobe_lr               object
lobe          

In [21]:
# Step 3: Iterate over each row in the DataFrame to read the Visium datasets
adata_list = []
for i in range(len(metadata_df)):
    # Extract the directory path and metadata information
    irow = metadata_df.iloc[i] # get the i-th row
    
    # Step 4: Read the Visium dataset using Scanpy
    adata = sq.read.visium(path=irow['dir'], counts_file = "filtered_feature_bc_matrix.h5", library_id=irow['library_id'])

    # https://stackoverflow.com/questions/74893175/trying-to-concat-a-list-of-12-anndata-objects-but-am-getting-duplicates
    # dont run this because we are using ENSEMBL ID?
    adata.var_names_make_unique() 
    

    # Change gene symbol to ENSEMBL ID
    # adata.var['symbol'] = adata.var_names
    # adata.var.set_index('gene_ids', drop = True, inplace=True)
    
    # Step 5: Add the other metadata columns to the AnnData object
    # You can add any other columns as metadata (e.g., row['other_column_name'])
    for colname in metadata_df.columns:
        if colname not in ['dir','source_image_path']:  # Skip the 'dir' column
            adata.obs[colname] = irow[colname]
    
    # Give each adata a name (sample name)
    adata.name = irow['library_id']


    ## define genes
    # mitochondrial genes, "MT-" for human, "Mt-" for mouse
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    # ribosomal genes
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    # hemoglobin genes
    # .startswith() does not support regex
    adata.var["hb"] = adata.var_names.str.contains("^HB[ABDEGMQZ]")

    # Step 6: Append the AnnData object to the list
    adata_list.append(adata)

In [22]:

row = metadata_df.loc[metadata_df["library_id"] == adata_list[0].name]
if row["xmin"][0]!= 0:
    adata_list[0] = adata_list[0][adata_list[0].obsm["spatial"][:,1]>4000]

In [23]:
metadata_df.loc[metadata_df["library_id"] == adata.name]

,library_id,alignment,desired_rc,actual_rc,dir,xmin,xmax,ymin,ymax,hpx,vpx,slide,area,date,cytassist_ver,seq_batch,probe_date,probe_lot,probe_group,cond,cond_rep,rep,age,sex,bmi,packyears,FVC,DLCO,6MWD,outcome_long,tte_long,outcome_24m,tte_24m,lobe_lr,lobe,source_image_path
44,22_16220_B1,man,90,91,/home/ubuntu/jk2/scvi-tools/out_spaceranger401...,0,0,0,0,14927,15798,V44A10-302,A1,2024-12-17,2.0.1.3,5,2024-07-01,203947,2,IPF,IPF_10,10,58,F,26,45.0,64.0,39.0,NaN,ALIVE,34,ALIVE,24,right,middle,/home/ubuntu/jk2/scvi-tools/out_spaceranger401...


In [24]:
# Perform quality control metrics calculations and filtering
adata_list_filtered = []
filter_df = []
for adata in adata_list:
      ## This adds additional columns to adata.obs 
    # ex. n_genes_by_genes: number of genes detected for each cell
    # ex. total_genes: total read count for each cell
    # ex. pct for each of mt, ribo and hb
    sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
    )

    ## Filter based on coordinates
    # metadata has four columns (xmin, xmax, ymin and ymax)
    # 0 means no filtering is applied
    # Multiply these values by 25 to get pixel coordinates to filter spots
    adata_name = adata.name

    # Gets the row that mathces the library_id of the adata
    row = metadata_df.loc[metadata_df["library_id"] == adata_name]
    
    #
    if row["xmin"].iloc[0] != 0:
      # adata.obsm["spatial"] is a list of list of x and y coordiantes (i.e. [[X,Y],[X,Y]...])
      # [:,0] means get all [X,Y] lists then pick the 0th index of each, which is X
      # > for greater than "xmin"
      adata = adata[adata.obsm["spatial"][:,0] > 25*adata.obs["xmin"]].copy()

    if row["xmax"].iloc[0] != 0:
      # < for less than "xmax"
      adata = adata[adata.obsm["spatial"][:,0] < 25*adata.obs["xmax"]].copy()

    if row["ymin"].iloc[0] != 0:
      # [:,1] means get all [X,Y] lists then pick the 1th index of each, which is Y
      adata = adata[adata.obsm["spatial"][:,1] > 25*adata.obs["ymin"]].copy()

    if row["ymax"].iloc[0] != 0:
      adata = adata[adata.obsm["spatial"][:,1] < 25*adata.obs["ymax"]].copy()


    ### Obs (spots/cells) filter
    # Store the number of obs before read/gene count filtering
    obs_original = adata.n_obs

    # # adata.X is sparse, so convert to dense
    # data_matrix = adata.X.toarray()
    # # flatten
    # flattened_data = data_matrix.flatten()

    # Calculate various count quantiles using numpy
    # First make a list of quantiles to calculate
    count_quant = [0.01, 0.10, 0.25, 0.5, 0.75, 0.90, 0.99]

    count_quant_np = np.quantile(adata.obs.loc[:,"total_counts"], count_quant)
    

    # Apply filter based on min and max counts
    sc.pp.filter_cells(adata, min_counts=count_quant_np[0], inplace= True)
    #sc.pp.filter_cells(adata, max_counts=count_quant_np[-1], inplace= True)

    # Calculate various gene quantiles using numpy
    gene_quant = [0.10, 0.25, 0.5, 0.75, 0.90]
    gene_quant_np = np.quantile(adata.obs.loc[:,"n_genes_by_counts"], gene_quant)

    sc.pp.filter_cells(adata, min_genes = 100, inplace = True)



 

    # Store the number of obs after filtering
    obs_count_filter = adata.n_obs

    # Calculate the number of obs removed
    obs_removed = obs_original - obs_count_filter





   

    # Calculate the number of obs removed by mito filter
    obs_removed_mito = adata[adata.obs["pct_counts_mt"]>=30].shape[0]


    # .copy() does not copy the name attribute
    #adata_name = adata.name

    # Only keep obs with <= 30% mito genes
    adata = adata[adata.obs["pct_counts_mt"]<30].copy()
    adata.name = adata_name


    # filter out UNC_6 because it has very few gene counts across spots
    if adata.name != "20_17478_A1":
      adata_list_filtered.append(adata)
    



    #Store the relevant information in a dictionary 
    filter_df.append({
        'name': adata.name,
        'min_count': count_quant_np[0],
        'countq10': count_quant_np[1],
        'countq25': count_quant_np[2],
        'countq50': count_quant_np[3],
        'countq75': count_quant_np[4],
        'countq90': count_quant_np[5],
        'max_count': count_quant_np[6],
        'obs_removed': obs_removed,
        'obs_remaining': obs_count_filter,


        'geneq10': gene_quant_np[0],
        'geneq25': gene_quant_np[1],
        'geneq50': gene_quant_np[2],
        'geneq75': gene_quant_np[3],
        'geneq90': gene_quant_np[4]
    })



In [25]:
# Save filter_df
# filter_df is a list-dictionary, so convert into pd.DataFrame
pd.DataFrame(pd.concat(
    [pd.DataFrame(filter_df).iloc[:,:1], 
     metadata_df[['seq_batch','cond','cond_rep']], 
     pd.DataFrame(filter_df).iloc[:,1:]], axis = 1)).to_excel("filter_df.xlsx", index = False)

In [26]:
# Concat all adata into one big adata

adata_filtered_concat = sc.concat(adata_list_filtered, uns_merge="unique", join="outer", 
                                    index_unique= '-', keys=[adata.name for adata in adata_list_filtered])

In [27]:
# Copy ENSEMBL gene names
# The axis is sc.concat is by obs, so var is lost
adata_filtered_concat.var = pd.DataFrame(adata_list_filtered[0].var.iloc[:,0])

In [28]:
# Filter out spots whose .obsm['spatial'] arrays contain NaN
arr1 = ~np.isnan(adata_filtered_concat.obsm['spatial'])[:,0] # all lists and their first item (presumably x coordinate)
arr2 = ~np.isnan(adata_filtered_concat.obsm['spatial'])[:,1] # all lists and their second item (presumably y coordinate)
arr = arr1 & arr2
adata_filtered_concat = adata_filtered_concat[arr,:]

In [29]:
# Save raw .X and .var
adata_filtered_concat.raw = adata_filtered_concat.copy()

In [30]:
adata_filtered_concat.write('/home/ubuntu/jk2/scvi-tools/h5ad/adata_filtered_concat.h5ad')

In [31]:
# Concat all adata into one big adata UNFILTERED
adata_concat = sc.concat(adata_list, uns_merge = "unique", join = "outer",
                         index_unique="-", keys = [adata.name for adata in adata_list])

In [32]:
adata_concat.write('/home/ubuntu/jk2/scvi-tools/h5ad/adata_concat.h5ad')